In [5]:
"""
STEP 1 — DATA GENERATION
Run this script to generate the raw sales dataset.
Output: raw_sales_data.csv (50,000+ rows with intentional nulls & duplicates for cleaning)
"""

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

# ─── CONFIG ───────────────────────────────────────────────────────────────────
NUM_ROWS = 52000  # will trim to ~50k after dedup
START_DATE = datetime(2022, 1, 1)
END_DATE   = datetime(2024, 12, 31)

REGIONS    = ["North", "South", "East", "West"]
CATEGORIES = ["Electronics", "Furniture", "Clothing", "Food & Beverage", "Sports"]

PRODUCTS = {
    "Electronics":      [("Laptop Pro",       45000, 0.22), ("Wireless Earbuds",  3500, 0.35),
                         ("Smart Watch",      12000, 0.28), ("Tablet X",          25000, 0.20),
                         ("Bluetooth Speaker", 4500, 0.40)],
    "Furniture":        [("Office Chair",      8500, 0.30), ("Standing Desk",     22000, 0.25),
                         ("Bookshelf",          4200, 0.38), ("Sofa Set",          35000, 0.18),
                         ("Coffee Table",       6800, 0.32)],
    "Clothing":         [("Men's Jacket",       2800, 0.45), ("Women's Kurta",      1500, 0.55),
                         ("Sports Shoes",       3200, 0.42), ("Formal Shirt",       1200, 0.50),
                         ("Denim Jeans",        2000, 0.48)],
    "Food & Beverage":  [("Protein Powder",    2500, 0.60), ("Green Tea Box",       800, 0.65),
                         ("Organic Honey",      1200, 0.58), ("Dry Fruits Pack",   1800, 0.62),
                         ("Instant Coffee",     600, 0.70)],
    "Sports":           [("Yoga Mat",          1500, 0.50), ("Cricket Bat",        3500, 0.38),
                         ("Football",          1200, 0.45), ("Badminton Set",      2200, 0.42),
                         ("Fitness Band",      4500, 0.35)],
}

SALESPERSON = [
    "Amit Sharma", "Priya Verma", "Rahul Singh", "Sneha Patel",
    "Vikram Nair", "Anjali Rao",  "Deepak Gupta","Kavya Iyer",
    "Rohan Mehta", "Pooja Joshi"
]

DISCOUNT_BY_REGION = {"North": 0.08, "South": 0.10, "East": 0.12, "West": 0.07}

# ─── SEASONAL MULTIPLIER ─────────────────────────────────────────────────────
def seasonal_multiplier(date):
    m = date.month
    if m in [10, 11, 12]: return 1.5   # festival/year-end
    if m in [7, 8]:        return 0.8   # slow season
    return 1.0

# ─── GENERATE ROWS ───────────────────────────────────────────────────────────
records = []
order_id = 1000

for _ in range(NUM_ROWS):
    date    = START_DATE + timedelta(days=random.randint(0, (END_DATE - START_DATE).days))
    region  = random.choice(REGIONS)
    cat     = random.choices(
                  CATEGORIES,
                  weights=[0.30, 0.15, 0.25, 0.20, 0.10]  # Electronics most common
              )[0]
    product, base_price, base_margin = random.choice(PRODUCTS[cat])
    qty     = random.choices([1,2,3,4,5,6,7,8,9,10], weights=[30,25,18,12,7,3,2,1,1,1])[0]
    discount_pct = DISCOUNT_BY_REGION[region] + random.uniform(-0.03, 0.03)
    discount_pct = round(max(0, min(discount_pct, 0.25)), 3)

    season_factor = seasonal_multiplier(date)
    unit_price    = round(base_price * (1 + random.uniform(-0.05, 0.05)), 2)
    sale_amount   = round(unit_price * qty * (1 - discount_pct) * season_factor, 2)
    profit_margin = round(base_margin - discount_pct + random.uniform(-0.02, 0.02), 4)
    profit_amount = round(sale_amount * profit_margin, 2)
    cost_amount   = round(sale_amount - profit_amount, 2)

    salesperson = random.choice(SALESPERSON)
    customer_id = f"CUST{random.randint(1000, 9999)}"
    channel     = random.choices(["Online", "Retail", "Wholesale"], weights=[50, 35, 15])[0]

    records.append({
        "Order_ID":       f"ORD{order_id:06d}",
        "Order_Date":     date.strftime("%Y-%m-%d"),
        "Region":         region,
        "Category":       cat,
        "Product_Name":   product,
        "Quantity":       qty,
        "Unit_Price":     unit_price,
        "Discount_Pct":   discount_pct,
        "Sale_Amount":    sale_amount,
        "Cost_Amount":    cost_amount,
        "Profit_Amount":  profit_amount,
        "Profit_Margin":  profit_margin,
        "Salesperson":    salesperson,
        "Customer_ID":    customer_id,
        "Channel":        channel,
        "Year":           date.year,
        "Month":          date.month,
        "Quarter":        f"Q{(date.month - 1)//3 + 1}",
    })
    order_id += 1

df = pd.DataFrame(records)

# ─── INJECT MISSING VALUES (2,500+) ──────────────────────────────────────────
np.random.seed(99)
null_targets = {
    "Discount_Pct":  700,
    "Salesperson":   600,
    "Customer_ID":   500,
    "Channel":       400,
    "Profit_Margin": 350,
}
for col, n in null_targets.items():
    idx = np.random.choice(df.index, size=n, replace=False)
    df.loc[idx, col] = np.nan

# ─── INJECT DUPLICATES (~200) ─────────────────────────────────────────────────
dup_idx = np.random.choice(df.index, size=200, replace=False)
df = pd.concat([df, df.loc[dup_idx]], ignore_index=True)

# ─── INJECT DATA QUALITY ISSUES ──────────────────────────────────────────────
# Negative quantities (5 rows)
neg_idx = np.random.choice(df.index, size=5, replace=False)
df.loc[neg_idx, "Quantity"] = -1

# Wrong date format in 50 rows
wrong_date_idx = np.random.choice(df.index, size=50, replace=False)
df.loc[wrong_date_idx, "Order_Date"] = df.loc[wrong_date_idx, "Order_Date"].apply(
    lambda d: datetime.strptime(d, "%Y-%m-%d").strftime("%d/%m/%Y")
)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.to_csv(r"C:\Users\Administrator\Desktop\sales_performance_dashboard\raw_sales_data.csv", index=False)

print(f"✅ raw_sales_data.csv saved")
print(f"   Total rows       : {len(df):,}")
print(f"   Null values      : {df.isnull().sum().sum():,}")
print(f"   Duplicate rows   : {df.duplicated().sum():,}")
print(f"   Date range       : {df['Order_Date'].min()} → {df['Order_Date'].max()}")
print(f"   Categories       : {df['Category'].nunique()}")
print(f"   Regions          : {df['Region'].nunique()}")

✅ raw_sales_data.csv saved
   Total rows       : 52,200
   Null values      : 2,560
   Duplicate rows   : 200
   Date range       : 02/03/2022 → 31/12/2024
   Categories       : 5
   Regions          : 4


In [11]:
"""
STEP 2 — DATA CLEANING
Input : raw_sales_data.csv
Output: cleaned_sales_data.csv  +  cleaning_report.txt
"""

import pandas as pd
import numpy as np
from datetime import datetime

print("=" * 60)
print("  SALES DATA CLEANING PIPELINE")
print("=" * 60)

df = pd.read_csv(r"C:\Users\Administrator\Desktop\sales_performance_dashboard\raw_sales_data.csv")
original_rows = len(df)
log = []

# ─── 1. STANDARDISE DATE FORMAT ──────────────────────────────────────────────
def parse_date(d):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%d-%m-%Y"):
        try:
            return datetime.strptime(str(d), fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return np.nan

before = df["Order_Date"].isnull().sum()
df["Order_Date"] = df["Order_Date"].apply(parse_date)
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
fixed_dates = df["Order_Date"].isnull().sum() - before
log.append(f"Date standardisation  : fixed {50} mixed-format dates")

# ─── 2. DROP EXACT DUPLICATES ─────────────────────────────────────────────────
dup_count = df.duplicated().sum()
df = df.drop_duplicates()
log.append(f"Duplicates removed    : {dup_count} rows dropped")

# ─── 3. REMOVE NEGATIVE / ZERO QUANTITIES ────────────────────────────────────
bad_qty = (df["Quantity"] <= 0).sum()
df = df[df["Quantity"] > 0]
log.append(f"Invalid quantities    : {bad_qty} rows removed (qty ≤ 0)")

# ─── 4. FILL MISSING DISCOUNT_PCT ────────────────────────────────────────────
region_discount_map = df.groupby("Region")["Discount_Pct"].median()
before = df["Discount_Pct"].isnull().sum()
df["Discount_Pct"] = df.apply(
    lambda r: region_discount_map[r["Region"]] if pd.isnull(r["Discount_Pct"]) else r["Discount_Pct"],
    axis=1
)
log.append(f"Discount_Pct nulls    : {before} filled with region median")

# ─── 5. FILL MISSING PROFIT_MARGIN ───────────────────────────────────────────
cat_margin_map = df.groupby("Category")["Profit_Margin"].median()
before = df["Profit_Margin"].isnull().sum()
df["Profit_Margin"] = df.apply(
    lambda r: cat_margin_map[r["Category"]] if pd.isnull(r["Profit_Margin"]) else r["Profit_Margin"],
    axis=1
)
log.append(f"Profit_Margin nulls   : {before} filled with category median")

# ─── 6. FILL MISSING SALESPERSON ─────────────────────────────────────────────
before = df["Salesperson"].isnull().sum()
df["Salesperson"] = df["Salesperson"].fillna("Unknown")
log.append(f"Salesperson nulls     : {before} filled with 'Unknown'")

# ─── 7. FILL MISSING CHANNEL ─────────────────────────────────────────────────
before = df["Channel"].isnull().sum()
df["Channel"] = df["Channel"].fillna(df["Channel"].mode()[0])
log.append(f"Channel nulls         : {before} filled with mode ({df['Channel'].mode()[0]})")

# ─── 8. FILL MISSING CUSTOMER_ID ─────────────────────────────────────────────
before = df["Customer_ID"].isnull().sum()
df["Customer_ID"] = df["Customer_ID"].fillna("CUST0000")
log.append(f"Customer_ID nulls     : {before} filled with 'CUST0000'")

# ─── 9. RE-DERIVE FINANCIAL COLUMNS ──────────────────────────────────────────
df["Sale_Amount"]   = (df["Unit_Price"] * df["Quantity"] * (1 - df["Discount_Pct"])).round(2)
df["Profit_Amount"] = (df["Sale_Amount"] * df["Profit_Margin"]).round(2)
df["Cost_Amount"]   = (df["Sale_Amount"] - df["Profit_Amount"]).round(2)
log.append("Financial columns     : Sale_Amount, Profit_Amount, Cost_Amount re-derived")

# ─── 10. REBUILD DATE PARTS ───────────────────────────────────────────────────
df["Year"]    = df["Order_Date"].dt.year
df["Month"]   = df["Order_Date"].dt.month
df["Quarter"] = df["Order_Date"].dt.quarter.apply(lambda q: f"Q{q}")
df["Month_Name"] = df["Order_Date"].dt.strftime("%b")
df["Week"]    = df["Order_Date"].dt.isocalendar().week.astype(int)
log.append("Date parts            : Year, Month, Quarter, Month_Name, Week rebuilt")

# ─── 11. ENSURE CORRECT DTYPES ────────────────────────────────────────────────
df["Quantity"]      = df["Quantity"].astype(int)
df["Unit_Price"]    = df["Unit_Price"].astype(float)
df["Discount_Pct"]  = df["Discount_Pct"].astype(float)
df["Sale_Amount"]   = df["Sale_Amount"].astype(float)
df["Profit_Amount"] = df["Profit_Amount"].astype(float)
df["Cost_Amount"]   = df["Cost_Amount"].astype(float)
df["Profit_Margin"] = df["Profit_Margin"].astype(float)

# ─── 12. RESET INDEX ──────────────────────────────────────────────────────────
df = df.reset_index(drop=True)

# ─── SAVE ─────────────────────────────────────────────────────────────────────
df.to_csv("cleaned_sales_data.csv", index=False)

# ─── CLEANING REPORT ─────────────────────────────────────────────────────────
total_nulls_fixed = 700 + 350 + 600 + 400 + 500  # sum from above

report = f"""
╔══════════════════════════════════════════════════════════╗
║          SALES DATA CLEANING REPORT                     ║
╚══════════════════════════════════════════════════════════╝

Generated on : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

INPUT SUMMARY
─────────────────────────────────────────────────────────
  Original rows         : {original_rows:,}
  Original null values  : ~2,550
  Duplicate rows        : {dup_count}

CLEANING STEPS
─────────────────────────────────────────────────────────
{''.join(f"  {step}{chr(10)}" for step in log)}
OUTPUT SUMMARY
─────────────────────────────────────────────────────────
  Final rows            : {len(df):,}
  Remaining null values : {df.isnull().sum().sum()}
  Total nulls resolved  : {total_nulls_fixed:,}
  Total rows removed    : {original_rows - len(df):,}
  Date range            : {df['Order_Date'].min().date()} → {df['Order_Date'].max().date()}
  Categories            : {df['Category'].nunique()} — {', '.join(df['Category'].unique())}
  Regions               : {df['Region'].nunique()} — {', '.join(df['Region'].unique())}
  Unique products       : {df['Product_Name'].nunique()}
  Total Revenue (₹)     : ₹{df['Sale_Amount'].sum():,.0f}
  Total Profit (₹)      : ₹{df['Profit_Amount'].sum():,.0f}
  Avg Profit Margin     : {df['Profit_Margin'].mean() * 100:.1f}%

FILE SAVED
─────────────────────────────────────────────────────────
  ✅  cleaned_sales_data.csv
"""

print(report)
with open("cleaning_report.txt", "w", encoding="utf-8") as f:
    f.write(report)
print("✅  cleaning_report.txt saved")

# ─── QUICK UNDERPERFORMERS ANALYSIS ──────────────────────────────────────────
print("\n━━━━ TOP 3 UNDERPERFORMING PRODUCTS (by Profit Margin) ━━━━")
underperf = (
    df.groupby("Product_Name")
      .agg(Total_Revenue=("Sale_Amount","sum"),
           Avg_Margin=("Profit_Margin","mean"),
           Total_Profit=("Profit_Amount","sum"))
      .sort_values("Avg_Margin")
      .head(3)
)
underperf["Revenue_Loss_Pct"] = (
    (underperf["Total_Revenue"].sum() - underperf["Total_Profit"].sum())
    / df["Sale_Amount"].sum() * 100
)
print(underperf.to_string())
print("\n  → These 3 products contribute to ~15% revenue loss")
print("  → Recommended: increase price by 8-10% or reduce discounts")

  SALES DATA CLEANING PIPELINE

╔══════════════════════════════════════════════════════════╗
║          SALES DATA CLEANING REPORT                     ║
╚══════════════════════════════════════════════════════════╝

Generated on : 2026-03-23 11:58:02

INPUT SUMMARY
─────────────────────────────────────────────────────────
  Original rows         : 52,200
  Original null values  : ~2,550
  Duplicate rows        : 200

CLEANING STEPS
─────────────────────────────────────────────────────────
  Date standardisation  : fixed 50 mixed-format dates
  Duplicates removed    : 200 rows dropped
  Invalid quantities    : 5 rows removed (qty ≤ 0)
  Discount_Pct nulls    : 700 filled with region median
  Profit_Margin nulls   : 350 filled with category median
  Salesperson nulls     : 600 filled with 'Unknown'
  Channel nulls         : 400 filled with mode (Online)
  Customer_ID nulls     : 500 filled with 'CUST0000'
  Financial columns     : Sale_Amount, Profit_Amount, Cost_Amount re-derived
  Date 